## Rotate Developer Credential Keys

#### import modules

In [ ]:
from arcgis.gis import GIS
from json import dump, load
from datetime import datetime, timedelta
from pathlib import Path

#### helper functions

In [ ]:
def slot_for_key(key:str):
    slot = int(key[-10:-9])
    if slot == 1 or slot == 2:
        return slot

    return None

In [ ]:
class cfg:
    item_id = "4f26fde50fda40678d98c575031ee720"
    days_in_future=3
    config_file_path=Path.cwd() / "app-config.json"

#### connect to our portal

In [ ]:
# portal = GIS("home")
portal = GIS(profile="DTS2026")
print(f"connected as {portal.properties.user.username} to {portal.properties.name}")

#### Read the current API key from our file

In [ ]:
app_config_path = cfg.config_file_path
with open(app_config_path,"r") as file:
    data=load(file)

In [ ]:
current_api_key = data["apiKey"]

In [ ]:
current_api_key_slot = slot_for_key(current_api_key)

In [ ]:
print(f"current key is in slot {current_api_key_slot}")

#### Create a new expiration date

In [ ]:
new_expiration = (datetime.now()+timedelta(days=3)).replace(hour=23, minute=59, microsecond=999000)
print(f"new expiration date: {new_expiration.strftime("%c")}")

#### get the current developer credential item

In [ ]:
developer_credential_item = portal.admin.developer_credentials.get("4f26fde50fda40678d98c575031ee720")
developer_credential_item._item

#### create a new token in the open slot

In [ ]:
new_slot = 2 if current_api_key_slot == 1 else 1

In [ ]:
new_key = developer_credential_item.generate_token(slot=new_slot, expiration=new_expiration)
new_key_token = new_key.get("access_token")

In [ ]:
print(f"new key in slot {new_slot} created {new_key_token[-12:]}...")

#### write the new key to our app file

In [ ]:
data["apiKey"] = new_key_token

In [ ]:
print("writing new key to file...")
with open(app_config_path,"w") as f:
    dump(data, f, indent=2)
    print(f"\tnew key written to {app_config_path}")

#### invalidate the old API Key

In [ ]:
developer_credential_item.revoke(current_api_key_slot)

In [ ]:
print(f"old key in slot {current_api_key_slot} revoked")